### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [1]:
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [2]:
## step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)


In [3]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [4]:
### step 2: Vector Store
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding_model)

## step 3:MMR Retriever
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001634081F390>, search_type='mmr', search_kwargs={'k': 5})

In [5]:
## step 4 : LLM and Prompt

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

llm=init_chat_model("openai:o4-mini")
llm


ChatOpenAI(profile={'max_input_tokens': 200000, 'max_output_tokens': 100000, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000163D9462F50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000163D998C710>, root_client=<openai.OpenAI object at 0x00000163D92D3AD0>, root_async_client=<openai.AsyncOpenAI object at 0x00000163D998C210>, model_name='o4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [6]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatOpenAI(profile={'max_input_tokens': 200000, 'max_output_tokens': 100000, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000163D9462F50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000163D998C710>, root_client=<openai.OpenAI object at 0x00000163D92D3AD0>, root_async_client=<openai

In [12]:
print(query_expansion_chain.invoke({"query":"Langchain memory"}))

Expanded query:  
LangChain memory management OR “Lang Chain memory” OR “stateful LLMChain context retention” OR “conversation memory” OR “persistent session memory” OR “prompt context caching” OR “retrieval-augmented generation (RAG) memory” OR “MemoryModule” OR “ConversationBufferMemory” OR “ConversationBufferWindowMemory” OR “ChatMessageHistory” OR “VectorStoreMemory” OR “RedisMemory” OR “SQLiteMemory” OR “FAISS memory store” OR “Pinecone memory backend” OR “Weaviate vector store” OR “LLM context window” OR “chatbot conversation memory modules”


In [8]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [ ]:
# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        # lambda to map input to both "input" and "context" since runnable map requires all keys to be mapped as a function which is invoked dynamically with input
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

In [10]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:

{'input': "LangChain supported memory types and modules – what kinds of memory does the framework provide (e.g. short-term vs. long-term memory, session memory, persistent vs. ephemeral memory, conversational buffer, summary memory, combined memory)?  Include technical synonyms and context such as ConversationBufferMemory, ConversationSummaryMemory, CombinedMemory; memory backends and storage options like in-memory dict, JSON/filesystem, SQL/SQLite/Postgres, NoSQL/MongoDB, Redis; vector-store-based memory (FAISS, Pinecone, Weaviate, Milvus); and general context-management, state-persistence and retrieval-augmented design patterns in LangChain."}
✅ Answer:
 LangChain currently provides two built-in memory types:  
  • ConversationBufferMemory – keeps a running buffer of the full chat history  
  • ConversationSummaryMemory – maintains a condensed summary of past turns to stay within token limits


In [11]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:

(“CrewAI agents” OR “Crew AI agents” OR “AI crew agents” OR “virtual crew assistant” OR “digital workforce agent” OR “intelligent crew management agent” OR “autonomous crew scheduling agent” OR “smart crew operations platform” OR “machine-learning crew optimization” OR “AI-driven crew dispatch system”)  
AND  
(crew OR team OR staff OR workforce OR personnel)  
AND  
(“artificial intelligence” OR AI OR “machine learning” OR “deep learning” OR autonomous OR intelligent)  
AND  
(agent OR assistant OR bot OR module OR system OR platform)  
AND  
(scheduling OR optimization OR management OR dispatch OR “task allocation” OR coordination OR automation OR planning OR operations OR support)  
AND  
(“multi-agent system” OR reinforcement OR “knowledge-based” OR NLP OR “natural language processing”)
✅ Answer:
 CrewAI agents are specialized autonomous sub-agents configured to work together in a structured workflow. Key characteristics include:

• Defined Role  
  – Each agent is